In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

______________________ من هنا يبدا الكود النهائي __________________________

إعداد + قراءة ملفات المرضى و KB 

تنظيف الأدوية


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# ==== Paths ====
PATIENTS_PATH = "/kaggle/input/clear-patient-files/1000_PURE_GENERIC.parquet"
KB_PATH       = "/kaggle/input/kb-openfda/kb_openfda.parquet"

# ==== Load patients ====
patients_df = pd.read_parquet(PATIENTS_PATH)
print("Patients:", patients_df.shape)
print(patients_df.columns.tolist())

# ==== Load KB ====
kb_df = pd.read_parquet(KB_PATH)
print("KB:", kb_df.shape)
print(kb_df.columns.tolist())


In [ ]:
STOPWORDS = {
    # dosing / frequency
    "bid","tid","qid","qhs","prn","stat","daily","day","ih","qh","qpm","qam",
    # forms / routes
    "oral","po","iv","im","sc","inhalation","inh","nas","nasal",
    "tab","tabs","tablet","cap","caps","capsule","puff","spray","patch",
    "bag","unit","units","actuation",
    # units
    "mg","mcg","g","ml","l",
    # common noise phrases
    "preadmission","medication","list","complete","accurate",
    # symptoms / non-drugs
    "wheezing","pain","fever","nausea","sob",
    # extra
    "dose"
}

def clean_processed_generics(meds):
    if meds is None:
        return []
    #  list تحويل الى  ndarray
    if isinstance(meds, np.ndarray):
        meds = meds.tolist()
    elif not isinstance(meds, (list, tuple)):
        try:
            meds = list(meds)
        except Exception:
            meds = [meds]

    out = []
    for m in meds:
        s = str(m).strip().lower()
        if not s or s == "nan":
            continue
        s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
        tokens = [t for t in s.split() if t and t not in STOPWORDS]
        if tokens:
            drug = " ".join(tokens).strip()
            if len(drug) >= 4:
                out.append(drug)

    # unique مع الحفاظ على الترتيب
    seen = set()
    cleaned = []
    for d in out:
        if d not in seen:
            seen.add(d)
            cleaned.append(d)
    return cleaned

final_patients = patients_df[["subject_id","diagnosis_names","processed_generic_meds"]].copy()
final_patients["clean_generics"] = final_patients["processed_generic_meds"].apply(clean_processed_generics)

# تأكد
lens = final_patients["clean_generics"].apply(len)
print(lens.describe())
print("Rows with meds >0:", (lens>0).sum(), "/", len(lens))

final_patients = final_patients[["subject_id","clean_generics","diagnosis_names"]].copy()
final_patients.head(3)


تجهيز KB  

In [ ]:
kb = kb_df[["primary_generic_name","conflict_type","text"]].copy()
kb = kb.rename(columns={
    "primary_generic_name": "drug",
    "conflict_type": "type",
    "text": "evidence"
})

kb["drug"] = kb["drug"].astype(str).str.lower().str.strip()
kb["evidence"] = kb["evidence"].astype(str)
kb = kb.dropna(subset=["drug","evidence"])

# اجمع النصوص لكل drug (أسرع بكثير من المرور على صفوف كثيرة)
kb_text = kb.groupby("drug")["evidence"].apply(lambda s: "\n".join(s.tolist())).to_dict()
kb_type = kb.groupby("drug")["type"].agg(lambda s: list(pd.unique(s))).to_dict()

print("KB indexed drugs:", len(kb_text))


دوال المطابقة + استخراج التعارضات لكل مريض (1000 مريض)


أرقام إجمالية (اللي طلبتها) + Unique (بدون تكرار)

In [ ]:
def to_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    try:
        return list(x)
    except Exception:
        return [x]

def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def match_any_phrases_fast(text, phrases, max_hits=8):
    txt = norm(text)
    hits = []
    for p in phrases:
        p2 = norm(p)
        if not p2:
            continue
        if p2 in txt:
            hits.append(p2)
            if len(hits) >= max_hits:
                break
    return hits

def conflicts_for_patient_fast(clean_meds, dx_list, kb_text, kb_type):
    meds = [norm(m) for m in to_list(clean_meds) if norm(m)]
    dxs  = [norm(d) for d in to_list(dx_list) if norm(d)]
    dxs  = [d for d in dxs if len(d) >= 4]  # تقليل ضجيج التشخيصات

    ddi_hits, dci_hits = [], []

    for drug in meds:
        txt = kb_text.get(drug, "")
        if not txt:
            continue

        other_meds = [m for m in meds if m != drug]

        # DDI: النص يذكر دواء ثاني من نفس قائمة المريض
        mentioned_meds = match_any_phrases_fast(txt, other_meds, max_hits=8)
        if mentioned_meds:
            ddi_hits.append({
                "drug": drug,
                "with": mentioned_meds,
                "type": kb_type.get(drug, [])
            })

        # DCI: النص يذكر تشخيص من تشخيصات المريض
        mentioned_dx = match_any_phrases_fast(txt, dxs, max_hits=8)
        if mentioned_dx:
            dci_hits.append({
                "drug": drug,
                "dx": mentioned_dx,
                "type": kb_type.get(drug, [])
            })

    return ddi_hits, dci_hits

# ==== Run on all patients ====
results = []
for _, row in final_patients.iterrows():
    sid  = int(row["subject_id"])
    meds = row["clean_generics"]
    dxs  = row["diagnosis_names"]

    ddi, dci = conflicts_for_patient_fast(meds, dxs, kb_text, kb_type)

    results.append({
        "subject_id": sid,
        "n_meds": len(to_list(meds)),
        "n_dx": len(to_list(dxs)),
        "ddi_count": len(ddi),
        "dci_count": len(dci),
        "ddi_hits": ddi,   # خله كامل أو قصّه لو تبي
        "dci_hits": dci,
    })

conflicts_df = pd.DataFrame(results)
conflicts_df.head(3)


In [ ]:
# ---- counts على مستوى المرضى ----
n_patients = len(conflicts_df)
n_any_ddi = int((conflicts_df["ddi_count"] > 0).sum())
n_any_dci = int((conflicts_df["dci_count"] > 0).sum())
n_any_conflict = int(((conflicts_df["ddi_count"] > 0) | (conflicts_df["dci_count"] > 0)).sum())
n_no_conflict = int(((conflicts_df["ddi_count"] == 0) & (conflicts_df["dci_count"] == 0)).sum())

total_ddi = int(conflicts_df["ddi_count"].sum())
total_dci = int(conflicts_df["dci_count"].sum())

print("Patients total:", n_patients)
print("Patients with any DDI:", n_any_ddi)
print("Patients with any DCI:", n_any_dci)
print("Patients with any conflict (DDI or DCI):", n_any_conflict)
print("Patients with no conflict:", n_no_conflict)
print("\nTotal DDI hits (raw):", total_ddi)
print("Total DCI hits (raw):", total_dci)
print("Total hits (raw):", total_ddi + total_dci)

# ---- Unique تعارضات  ----
def ddi_unique_count(ddi_hits):
    pairs = set()
    for h in (ddi_hits or []):
        a = norm(h.get("drug",""))
        for b in h.get("with", []):
            b = norm(b)
            if a and b:
                pairs.add(tuple(sorted([a,b])))
    return len(pairs)

def dci_unique_count(dci_hits):
    pairs = set()
    for h in (dci_hits or []):
        a = norm(h.get("drug",""))
        for d in h.get("dx", []):
            d = norm(d)
            if a and d:
                pairs.add((a,d))
    return len(pairs)

conflicts_df["ddi_unique"] = conflicts_df["ddi_hits"].apply(ddi_unique_count)
conflicts_df["dci_unique"] = conflicts_df["dci_hits"].apply(dci_unique_count)

print("\nTotal unique DDI:", int(conflicts_df["ddi_unique"].sum()))
print("Total unique DCI:", int(conflicts_df["dci_unique"].sum()))
print("Patients with any unique DDI:", int((conflicts_df["ddi_unique"]>0).sum()))
print("Patients with any unique DCI:", int((conflicts_df["dci_unique"]>0).sum()))


حفظ النتائج (CSV + Parquet)

In [ ]:
OUT_PARQUET = "/kaggle/working/conflicts_1000.parquet"
OUT_CSV     = "/kaggle/working/conflicts_1000.csv"

conflicts_df.to_parquet(OUT_PARQUET, index=False)
conflicts_df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_PARQUET)
print("Saved:", OUT_CSV)


In [ ]:
import os
import re
import ast
import numpy as np
import pandas as pd
import gradio as gr

# =========================================================
# 1) Utils: find KB file (works on Kaggle + Local)
# =========================================================
def find_kb_file(filename="kb_openfda.parquet"):
    # Kaggle input datasets
    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for ds in os.listdir(kaggle_root):
            cand = os.path.join(kaggle_root, ds, filename)
            if os.path.exists(cand):
                return cand

    # Local: look in ./kb/
    local_cand = os.path.join(os.path.dirname(__file__), "kb", filename)
    if os.path.exists(local_cand):
        return local_cand

    # Local: current working directory
    cwd_cand = os.path.join(os.getcwd(), filename)
    if os.path.exists(cwd_cand):
        return cwd_cand

    return None

KB_PATH = find_kb_file()
if KB_PATH is None:
    raise FileNotFoundError(
        "❌ Cannot find kb_openfda.parquet.\n"
        "- On Kaggle: add your KB dataset and ensure it contains kb_openfda.parquet\n"
        "- Locally: place it in ./kb/kb_openfda.parquet (next to this script)\n"
    )

# =========================================================
# 2) Your cleaning logic (regex + stopwords)
# =========================================================
STOPWORDS = {
    "bid","tid","qid","qhs","prn","stat","daily","day","ih","qh","qpm","qam",
    "oral","po","iv","im","sc","inhalation","inh","nas","nasal",
    "tab","tabs","tablet","cap","caps","capsule","puff","spray","patch",
    "bag","unit","units","actuation",
    "mg","mcg","g","ml","l",
    "preadmission","medication","list","complete","accurate",
    "wheezing","pain","fever","nausea","sob",
    "dose"
}

def to_list(x):
    """Convert values to python list safely (supports stringified lists)."""
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # If it looks like a list/tuple in text
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple, np.ndarray)):
                    return list(v)
                return [v]
            except Exception:
                return [x]
        return [x]

    try:
        return list(x)
    except Exception:
        return [x]

def clean_processed_generics(meds):
    meds = to_list(meds)
    out = []

    for m in meds:
        s = str(m).strip().lower()
        if not s or s == "nan":
            continue

        s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
        tokens = [t for t in s.split() if t and t not in STOPWORDS]
        if tokens:
            drug = " ".join(tokens).strip()
            if len(drug) >= 4:
                out.append(drug)

    # unique while keeping order
    seen = set()
    cleaned = []
    for d in out:
        if d not in seen:
            seen.add(d)
            cleaned.append(d)
    return cleaned

def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def match_any_phrases_fast(text, phrases, max_hits=8):
    txt = norm(text)
    hits = []
    for p in phrases:
        p2 = norm(p)
        if not p2:
            continue
        if p2 in txt:
            hits.append(p2)
            if len(hits) >= max_hits:
                break
    return hits

# =========================================================
# 3) Load KB + build index (once)
# =========================================================
kb_df = pd.read_parquet(KB_PATH)

def build_kb_index(kb_df):
    kb = kb_df[["primary_generic_name","conflict_type","text"]].copy()
    kb = kb.rename(columns={
        "primary_generic_name": "drug",
        "conflict_type": "type",
        "text": "evidence"
    })

    kb["drug"] = kb["drug"].astype(str).str.lower().str.strip()
    kb["evidence"] = kb["evidence"].astype(str)
    kb = kb.dropna(subset=["drug","evidence"])

    kb_text = kb.groupby("drug")["evidence"].apply(lambda s: "\n".join(s.tolist())).to_dict()
    kb_type = kb.groupby("drug")["type"].agg(lambda s: list(pd.unique(s))).to_dict()
    return kb_text, kb_type

KB_TEXT, KB_TYPE = build_kb_index(kb_df)

# =========================================================
# 4) Conflict logic for ONE patient (your same approach)
# =========================================================
def conflicts_for_patient(clean_meds, dx_list):
    meds = [norm(m) for m in to_list(clean_meds) if norm(m)]
    dxs  = [norm(d) for d in to_list(dx_list) if norm(d)]
    dxs  = [d for d in dxs if len(d) >= 4]

    ddi_hits, dci_hits = [], []

    for drug in meds:
        txt = KB_TEXT.get(drug, "")
        if not txt:
            continue

        other_meds = [m for m in meds if m != drug]

        # DDI
        mentioned_meds = match_any_phrases_fast(txt, other_meds, max_hits=8)
        if mentioned_meds:
            ddi_hits.append({
                "interaction_type": "DDI",
                "drug": drug,
                "with": mentioned_meds,
                "kb_types_for_drug": KB_TYPE.get(drug, [])  # اختياري للاطلاع فقط
            })

        # DCI
        mentioned_dx = match_any_phrases_fast(txt, dxs, max_hits=8)
        if mentioned_dx:
        dci_hits.append({
            "interaction_type": "DCI",
            "drug": drug,
            "dx": mentioned_dx,
            "kb_types_for_drug": KB_TYPE.get(drug, [])  # اختياري
        })

    return ddi_hits, dci_hits, meds, dxs

# =========================================================
# 5) Read pharmacist file (CSV / Parquet)
# =========================================================
def read_patient_file(path):
    ext = os.path.splitext(path.lower())[1]
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError("Only CSV or Parquet allowed")

# =========================================================
# 6) Gradio handler
# =========================================================
REQUIRED_COLS = {"subject_id", "processed_generic_meds", "diagnosis_names"}

def analyze(file_obj, subject_id):
    if file_obj is None:
        return "❌ ارفع ملف الصيدلي أولًا", None, None, None

    # file_obj can be a temp file path
    path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)

    try:
        df = read_patient_file(path)
    except Exception as e:
        return f"❌ فشل قراءة الملف: {e}", None, None, None

    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        return f"❌ ملف الصيدلي ناقص أعمدة: {sorted(missing)}", None, None, None

    # validate subject_id
    try:
        sid = int(str(subject_id).strip())
    except Exception:
        return "❌ subject_id لازم يكون رقم", None, None, None

    sub = df.loc[df["subject_id"] == sid]
    if sub.empty:
        return f"❌ ما لقيت subject_id = {sid} داخل الملف", None, None, None

    row = sub.iloc[0]
    clean_meds = clean_processed_generics(row["processed_generic_meds"])
    ddi_hits, dci_hits, meds_normed, dxs_normed = conflicts_for_patient(clean_meds, row["diagnosis_names"])

    # Make tables
    ddi_rows = []
    for h in ddi_hits:
        ddi_rows.append({
            "drug": h.get("drug",""),
            "with": ", ".join(h.get("with", [])),
            "type": ", ".join(h.get("type", []))
        })
    dci_rows = []
    for h in dci_hits:
        dci_rows.append({
            "drug": h.get("drug",""),
            "dx": ", ".join(h.get("dx", [])),
            "type": ", ".join(h.get("type", []))
        })

    ddi_table = pd.DataFrame(ddi_rows) if ddi_rows else pd.DataFrame(columns=["drug","with","type"])
    dci_table = pd.DataFrame(dci_rows) if dci_rows else pd.DataFrame(columns=["drug","dx","type"])

    summary = (
        f"✅ KB loaded from: {KB_PATH}\n"
        f"KB indexed drugs: {len(KB_TEXT)}\n\n"
        f"Patient subject_id: {sid}\n"
        f"Clean meds count: {len(clean_meds)}\n"
        f"DX count: {len(to_list(row['diagnosis_names']))}\n"
        f"DDI hits: {len(ddi_hits)}\n"
        f"DCI hits: {len(dci_hits)}\n"
    )

    meds_text = "\n".join(clean_meds) if clean_meds else "(no meds after cleaning)"
    dx_text   = "\n".join(dxs_normed) if dxs_normed else "(no dx)"

    return summary, meds_text, dx_text, ddi_table, dci_table

# =========================================================
# 7) UI
# =========================================================
with gr.Blocks(title="MedGuard - Conflicts Viewer") as demo:
    gr.Markdown("## MedGuard – رفع ملف الصيدلي وعرض تعارضات مريض واحد")
    gr.Markdown(
        f"**KB_PATH:** `{KB_PATH}`  \n"
        f"**Indexed drugs:** `{len(KB_TEXT)}`  \n"
        "الملف المطلوب لازم يحتوي الأعمدة: `subject_id`, `processed_generic_meds`, `diagnosis_names`"
    )

    with gr.Row():
        file_in = gr.File(label="ارفع ملف الصيدلي (CSV/Parquet)")
        sid_in  = gr.Textbox(label="subject_id", placeholder="مثال: 10000032")

    run_btn = gr.Button("تحليل")

    summary_out = gr.Textbox(label="Summary", lines=10)
    with gr.Row():
        meds_out = gr.Textbox(label="Clean meds (after regex)", lines=12)
        dx_out   = gr.Textbox(label="Diagnoses (normalized)", lines=12)

    gr.Markdown("### DDI (Drug–Drug)")
    ddi_out = gr.Dataframe(interactive=False)

    gr.Markdown("### DCI (Drug–Condition)")
    dci_out = gr.Dataframe(interactive=False)

    run_btn.click(
        fn=analyze,
        inputs=[file_in, sid_in],
        outputs=[summary_out, meds_out, dx_out, ddi_out, dci_out]
    )

import os
import re
import ast
import numpy as np
import pandas as pd
import gradio as gr

# =========================================================
# 1) Utils: find KB file (works on Kaggle + Local)
# =========================================================
def find_kb_file(filename="kb_openfda.parquet"):
    # Kaggle input datasets
    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for ds in os.listdir(kaggle_root):
            cand = os.path.join(kaggle_root, ds, filename)
            if os.path.exists(cand):
                return cand

    # Local: look in ./kb/
    local_cand = os.path.join(os.path.dirname(__file__), "kb", filename)
    if os.path.exists(local_cand):
        return local_cand

    # Local: current working directory
    cwd_cand = os.path.join(os.getcwd(), filename)
    if os.path.exists(cwd_cand):
        return cwd_cand

    return None

KB_PATH = find_kb_file()
if KB_PATH is None:
    raise FileNotFoundError(
        "❌ Cannot find kb_openfda.parquet.\n"
        "- On Kaggle: add your KB dataset and ensure it contains kb_openfda.parquet\n"
        "- Locally: place it in ./kb/kb_openfda.parquet (next to this script)\n"
    )

# =========================================================
# 2) Your cleaning logic (regex + stopwords)
# =========================================================
STOPWORDS = {
    "bid","tid","qid","qhs","prn","stat","daily","day","ih","qh","qpm","qam",
    "oral","po","iv","im","sc","inhalation","inh","nas","nasal",
    "tab","tabs","tablet","cap","caps","capsule","puff","spray","patch",
    "bag","unit","units","actuation",
    "mg","mcg","g","ml","l",
    "preadmission","medication","list","complete","accurate",
    "wheezing","pain","fever","nausea","sob",
    "dose"
}

def to_list(x):
    """Convert values to python list safely (supports stringified lists)."""
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # If it looks like a list/tuple in text
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple, np.ndarray)):
                    return list(v)
                return [v]
            except Exception:
                return [x]
        return [x]

    try:
        return list(x)
    except Exception:
        return [x]

def clean_processed_generics(meds):
    meds = to_list(meds)
    out = []

    for m in meds:
        s = str(m).strip().lower()
        if not s or s == "nan":
            continue

        s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
        tokens = [t for t in s.split() if t and t not in STOPWORDS]
        if tokens:
            drug = " ".join(tokens).strip()
            if len(drug) >= 4:
                out.append(drug)

    # unique while keeping order
    seen = set()
    cleaned = []
    for d in out:
        if d not in seen:
            seen.add(d)
            cleaned.append(d)
    return cleaned

def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def match_any_phrases_fast(text, phrases, max_hits=8):
    txt = norm(text)
    hits = []
    for p in phrases:
        p2 = norm(p)
        if not p2:
            continue
        if p2 in txt:
            hits.append(p2)
            if len(hits) >= max_hits:
                break
    return hits

# =========================================================
# 3) Load KB + build index (once)
# =========================================================
kb_df = pd.read_parquet(KB_PATH)

def build_kb_index(kb_df):
    kb = kb_df[["primary_generic_name","conflict_type","text"]].copy()
    kb = kb.rename(columns={
        "primary_generic_name": "drug",
        "conflict_type": "type",
        "text": "evidence"
    })

    kb["drug"] = kb["drug"].astype(str).str.lower().str.strip()
    kb["evidence"] = kb["evidence"].astype(str)
    kb = kb.dropna(subset=["drug","evidence"])

    kb_text = kb.groupby("drug")["evidence"].apply(lambda s: "\n".join(s.tolist())).to_dict()
    kb_type = kb.groupby("drug")["type"].agg(lambda s: list(pd.unique(s))).to_dict()
    return kb_text, kb_type

KB_TEXT, KB_TYPE = build_kb_index(kb_df)

# =========================================================
# 4) Conflict logic for ONE patient (your same approach)
# =========================================================
def conflicts_for_patient(clean_meds, dx_list):
    meds = [norm(m) for m in to_list(clean_meds) if norm(m)]
    dxs  = [norm(d) for d in to_list(dx_list) if norm(d)]
    dxs  = [d for d in dxs if len(d) >= 4]

    ddi_hits, dci_hits = [], []

    for drug in meds:
        txt = KB_TEXT.get(drug, "")
        if not txt:
            continue

        other_meds = [m for m in meds if m != drug]

        # DDI
        mentioned_meds = match_any_phrases_fast(txt, other_meds, max_hits=8)
        if mentioned_meds:
            ddi_hits.append({
                "drug": drug,
                "with": mentioned_meds,
                "type": KB_TYPE.get(drug, [])
            })

        # DCI
        mentioned_dx = match_any_phrases_fast(txt, dxs, max_hits=8)
        if mentioned_dx:
            dci_hits.append({
                "drug": drug,
                "dx": mentioned_dx,
                "type": KB_TYPE.get(drug, [])
            })

    return ddi_hits, dci_hits, meds, dxs

# =========================================================
# 5) Read pharmacist file (CSV / Parquet)
# =========================================================
def read_patient_file(path):
    ext = os.path.splitext(path.lower())[1]
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError("Only CSV or Parquet allowed")

# =========================================================
# 6) Gradio handler
# =========================================================
REQUIRED_COLS = {"subject_id", "processed_generic_meds", "diagnosis_names"}

def analyze(file_obj, subject_id):
    # لازم نرجع 5 outputs دائمًا
    empty_ddi = pd.DataFrame(columns=["drug","with","type"])
    empty_dci = pd.DataFrame(columns=["drug","dx","type"])

    if file_obj is None:
        return "❌ ارفع ملف الصيدلي أولًا", "", "", empty_ddi, empty_dci

    path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)

    try:
        df = read_patient_file(path)
    except Exception as e:
        return f"❌ فشل قراءة الملف: {e}", "", "", empty_ddi, empty_dci

    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        return f"❌ ملف الصيدلي ناقص أعمدة: {sorted(missing)}", "", "", empty_ddi, empty_dci

    # validate subject_id
    try:
        sid = int(str(subject_id).strip())
    except Exception:
        return "❌ subject_id لازم يكون رقم", "", "", empty_ddi, empty_dci

    sub = df.loc[df["subject_id"] == sid]
    if sub.empty:
        return f"❌ ما لقيت subject_id = {sid} داخل الملف", "", "", empty_ddi, empty_dci

    row = sub.iloc[0]
    clean_meds = clean_processed_generics(row["processed_generic_meds"])
    ddi_hits, dci_hits, meds_normed, dxs_normed = conflicts_for_patient(clean_meds, row["diagnosis_names"])

    ddi_rows = []
    for h in ddi_hits:
        ddi_rows.append({
            "drug": h.get("drug",""),
            "with": ", ".join(h.get("with", [])),
            "type": ", ".join(h.get("type", []))
        })
    dci_rows = []
    for h in dci_hits:
        dci_rows.append({
            "drug": h.get("drug",""),
            "dx": ", ".join(h.get("dx", [])),
            "type": ", ".join(h.get("type", []))
        })

    ddi_table = pd.DataFrame(ddi_rows) if ddi_rows else empty_ddi
    dci_table = pd.DataFrame(dci_rows) if dci_rows else empty_dci

    summary = (
        f"✅ KB loaded from: {KB_PATH}\n"
        f"KB indexed drugs: {len(KB_TEXT)}\n\n"
        f"Patient subject_id: {sid}\n"
        f"Clean meds count: {len(clean_meds)}\n"
        f"DX count: {len(to_list(row['diagnosis_names']))}\n"
        f"DDI hits: {len(ddi_hits)}\n"
        f"DCI hits: {len(dci_hits)}\n"
    )

    meds_text = "\n".join(clean_meds) if clean_meds else "(no meds after cleaning)"
    dx_text   = "\n".join(dxs_normed) if dxs_normed else "(no dx)"

    return summary, meds_text, dx_text, ddi_table, dci_table


# =========================================================
# 7) UI
# =========================================================
with gr.Blocks(title="MedGuard - Conflicts Viewer") as demo:
    gr.Markdown("## MedGuard – رفع ملف الصيدلي وعرض تعارضات مريض واحد")
    gr.Markdown(
        f"**KB_PATH:** `{KB_PATH}`  \n"
        f"**Indexed drugs:** `{len(KB_TEXT)}`  \n"
        "الملف المطلوب لازم يحتوي الأعمدة: `subject_id`, `processed_generic_meds`, `diagnosis_names`"
    )

    with gr.Row():
        file_in = gr.File(label="ارفع ملف الصيدلي (CSV/Parquet)")
        sid_in  = gr.Textbox(label="subject_id", placeholder="مثال: 10000032")

    run_btn = gr.Button("تحليل")

    summary_out = gr.Textbox(label="Summary", lines=10)
    with gr.Row():
        meds_out = gr.Textbox(label="Clean meds (after regex)", lines=12)
        dx_out   = gr.Textbox(label="Diagnoses (normalized)", lines=12)

    gr.Markdown("### DDI (Drug–Drug)")
    ddi_out = gr.Dataframe(interactive=False)

    gr.Markdown("### DCI (Drug–Condition)")
    dci_out = gr.Dataframe(interactive=False)

    run_btn.click(
        fn=analyze,
        inputs=[file_in, sid_in],
        outputs=[summary_out, meds_out, dx_out, ddi_out, dci_out]
    )

# لا تستعمل demo.queue()
demo.launch(share=True, debug=True, show_error=True)



In [ ]:
!pip -q install gradio pyarrow
!python medguard_ui.py


In [ ]:
import pandas as pd
import numpy as np
import ast
import os

# ========= عدّل اسم ملف الإدخال إذا لزم =========
INPUT_PATIENTS = "/kaggle/input/clear-patient-files/1000_PURE_GENERIC.parquet"
OUT_DIR = "/kaggle/working/sample_patients"
os.makedirs(OUT_DIR, exist_ok=True)

# ========= Load =========
df = pd.read_parquet(INPUT_PATIENTS)

# ========= تأكد من السكيما =========
REQUIRED = {"subject_id", "diagnosis_names", "processed_generic_meds"}
missing = REQUIRED - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# ========= دوال مساعدة =========
def to_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            if isinstance(v, (list, tuple)):
                return list(v)
        except Exception:
            return [x]
    return []

def has_content(x):
    return len(to_list(x)) > 0

# ========= فلترة مرضى مناسبين =========
candidates = df[
    df["processed_generic_meds"].apply(has_content) &
    df["diagnosis_names"].apply(has_content)
].copy()

print("Eligible patients:", len(candidates))

# ========= خذ أول 3 =========
sample = candidates.head(3)

# ========= احفظ كل مريض بملف =========
paths = []
for i, row in sample.iterrows():
    sid = row["subject_id"]
    out_path = f"{OUT_DIR}/patient_{sid}.parquet"

    one_patient = pd.DataFrame([{
        "subject_id": row["subject_id"],
        "diagnosis_names": row["diagnosis_names"],
        "processed_generic_meds": row["processed_generic_meds"],
    }])

    one_patient.to_parquet(out_path, index=False)
    paths.append(out_path)

paths


In [ ]:
!pip -q install gradio pyarrow transformers==4.50.0 accelerate bitsandbytes torch pillow huggingface_hub


In [ ]:
import os
import re
import ast
import json
import numpy as np
import pandas as pd
import gradio as gr

# ===== MedGemma imports =====
import torch
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image

# =========================================================
# 0) HuggingFace auth (Kaggle Secrets preferred)
# =========================================================
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token from Kaggle Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print("⚠️ Using HF_TOKEN from env (or empty)")

if HF_TOKEN:
    login(token=HF_TOKEN)

# =========================================================
# 1) Utils: find KB file (works on Kaggle + Local)
# =========================================================
def find_kb_file(filename="kb_openfda.parquet"):
    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for ds in os.listdir(kaggle_root):
            cand = os.path.join(kaggle_root, ds, filename)
            if os.path.exists(cand):
                return cand

    # Local fallbacks (if you ever run locally)
    local_cand = os.path.join(os.getcwd(), "kb", filename)
    if os.path.exists(local_cand):
        return local_cand

    cwd_cand = os.path.join(os.getcwd(), filename)
    if os.path.exists(cwd_cand):
        return cwd_cand

    return None

KB_PATH = find_kb_file()
if KB_PATH is None:
    raise FileNotFoundError(
        "❌ Cannot find kb_openfda.parquet.\n"
        "- On Kaggle: add your KB dataset and ensure it contains kb_openfda.parquet\n"
        "- Locally: place it in ./kb/kb_openfda.parquet\n"
    )

# =========================================================
# 2) Cleaning logic (regex + stopwords)
# =========================================================
STOPWORDS = {
    "bid","tid","qid","qhs","prn","stat","daily","day","ih","qh","qpm","qam",
    "oral","po","iv","im","sc","inhalation","inh","nas","nasal",
    "tab","tabs","tablet","cap","caps","capsule","puff","spray","patch",
    "bag","unit","units","actuation",
    "mg","mcg","g","ml","l",
    "preadmission","medication","list","complete","accurate",
    "wheezing","pain","fever","nausea","sob",
    "dose"
}

def to_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple, np.ndarray)):
                    return list(v)
                return [v]
            except Exception:
                return [x]
        return [x]

    try:
        return list(x)
    except Exception:
        return [x]

def clean_processed_generics(meds):
    meds = to_list(meds)
    out = []
    for m in meds:
        s = str(m).strip().lower()
        if not s or s == "nan":
            continue
        s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
        tokens = [t for t in s.split() if t and t not in STOPWORDS]
        if tokens:
            drug = " ".join(tokens).strip()
            if len(drug) >= 4:
                out.append(drug)

    seen = set()
    cleaned = []
    for d in out:
        if d not in seen:
            seen.add(d)
            cleaned.append(d)
    return cleaned

def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def match_any_phrases_fast(text, phrases, max_hits=8):
    txt = norm(text)
    hits = []
    for p in phrases:
        p2 = norm(p)
        if not p2:
            continue
        if p2 in txt:
            hits.append(p2)
            if len(hits) >= max_hits:
                break
    return hits

# =========================================================
# 3) Load KB + build index (once)
# =========================================================
kb_df = pd.read_parquet(KB_PATH)

def build_kb_index(kb_df):
    kb = kb_df[["primary_generic_name","conflict_type","text"]].copy()
    kb = kb.rename(columns={
        "primary_generic_name": "drug",
        "conflict_type": "type",
        "text": "evidence"
    })

    kb["drug"] = kb["drug"].astype(str).str.lower().str.strip()
    kb["evidence"] = kb["evidence"].astype(str)
    kb = kb.dropna(subset=["drug","evidence"])

    kb_text = kb.groupby("drug")["evidence"].apply(lambda s: "\n".join(s.tolist())).to_dict()
    kb_type = kb.groupby("drug")["type"].agg(lambda s: list(pd.unique(s))).to_dict()
    return kb_text, kb_type

KB_TEXT, KB_TYPE = build_kb_index(kb_df)
print("✅ KB indexed drugs:", len(KB_TEXT))

# =========================================================
# 4) Conflict logic for ONE patient
# =========================================================
def conflicts_for_patient(clean_meds, dx_list):
    meds = [norm(m) for m in to_list(clean_meds) if norm(m)]
    dxs  = [norm(d) for d in to_list(dx_list) if norm(d)]
    dxs  = [d for d in dxs if len(d) >= 4]

    ddi_hits, dci_hits = [], []

    for drug in meds:
        txt = KB_TEXT.get(drug, "")
        if not txt:
            continue

        other_meds = [m for m in meds if m != drug]

        mentioned_meds = match_any_phrases_fast(txt, other_meds, max_hits=8)
        if mentioned_meds:
            ddi_hits.append({
                "drug": drug,
                "with": mentioned_meds,
                "type": KB_TYPE.get(drug, [])
            })

        mentioned_dx = match_any_phrases_fast(txt, dxs, max_hits=8)
        if mentioned_dx:
            dci_hits.append({
                "drug": drug,
                "dx": mentioned_dx,
                "type": KB_TYPE.get(drug, [])
            })

    return ddi_hits, dci_hits, meds, dxs

# =========================================================
# 5) Read pharmacist file
# =========================================================
def read_patient_file(path):
    ext = os.path.splitext(path.lower())[1]
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError("Only CSV or Parquet allowed")

REQUIRED_COLS = {"subject_id", "processed_generic_meds", "diagnosis_names"}

# =========================================================
# 6) MedGemma load + generator (lazy load)
# =========================================================
MODEL_ID = "google/medgemma-1.5-4b-it"
medgemma_model = None
medgemma_processor = None
DUMMY_IMAGE = None

def load_medgemma_once():
    global medgemma_model, medgemma_processor, DUMMY_IMAGE

    if medgemma_model is not None:
        return True, "✅ MedGemma already loaded"

    if not torch.cuda.is_available():
        return False, "❌ GPU required for MedGemma on Kaggle"

    if not HF_TOKEN:
        return False, "❌ HF_TOKEN missing (add it in Kaggle Secrets as HF_TOKEN)"

    print("⏳ Loading MedGemma...")
    medgemma_processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
    medgemma_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN,
        low_cpu_mem_usage=True
    )
    medgemma_model.eval()

    DUMMY_IMAGE = Image.fromarray(np.ones((896, 896, 3), dtype=np.uint8) * 255)
    return True, "✅ MedGemma loaded"

def patient_prompt_ar(sid, clean_meds, dxs_normed, ddi_hits, dci_hits, max_ddi=3, max_dci=3):
    ddi = (ddi_hits or [])[:max_ddi]
    dci = (dci_hits or [])[:max_dci]

    prompt = f"""
أنت صيدلي سريري. اكتب تقرير انقليزي مختصر لمريض واحد بناءً على التعارضات المكتشفة من Knowledge Base.

معرف المريض: {sid}

الأدوية الحالية (Generics بعد التنظيف):
{json.dumps(clean_meds, ensure_ascii=False)}

التشخيصات/الحالات:
{json.dumps(dxs_normed, ensure_ascii=False)}

التعارضات المكتشفة:
DDI (دواء-دواء):
{json.dumps(ddi, ensure_ascii=False)}

DCI (دواء-حالة):
{json.dumps(dci, ensure_ascii=False)}

اكتب الإخراج بهذا الشكل فقط:
1) فقرة تلخص أهم المخاطر.
2) نقاط لكل تعارض مهم: (التعارض + السبب المختصر + الإجراء المقترح).
3) سطر أخير: "مستوى الخطورة العام: عالي/متوسط/منخفض"

قيود:
- لا تكتب Disclaimer.
- لا تخترع جرعات إذا غير موجودة.
"""
    return prompt.strip()

def medgemma_generate(prompt, max_new_tokens=260):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": DUMMY_IMAGE},
            {"type": "text", "text": prompt}
        ]
    }]

    inputs = medgemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(medgemma_model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = medgemma_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=1
        )

    text = medgemma_processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    del inputs, outputs
    torch.cuda.empty_cache()
    return text

# =========================================================
# 7) Main handler (conflicts + optional report)
# =========================================================
def analyze(file_obj, subject_id, use_medgemma):
    empty_ddi = pd.DataFrame(columns=["drug","with","type"])
    empty_dci = pd.DataFrame(columns=["drug","dx","type"])

    if file_obj is None:
        return "❌ ارفع ملف الصيدلي أولًا", "", "", empty_ddi, empty_dci, ""

    path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)

    try:
        df = read_patient_file(path)
    except Exception as e:
        return f"❌ فشل قراءة الملف: {e}", "", "", empty_ddi, empty_dci, ""

    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        return f"❌ ملف الصيدلي ناقص أعمدة: {sorted(missing)}", "", "", empty_ddi, empty_dci, ""

    try:
        sid = int(str(subject_id).strip())
    except Exception:
        return "❌ subject_id لازم يكون رقم", "", "", empty_ddi, empty_dci, ""

    sub = df.loc[df["subject_id"] == sid]
    if sub.empty:
        return f"❌ ما لقيت subject_id = {sid} داخل الملف", "", "", empty_ddi, empty_dci, ""

    row = sub.iloc[0]
    clean_meds = clean_processed_generics(row["processed_generic_meds"])
    ddi_hits, dci_hits, meds_normed, dxs_normed = conflicts_for_patient(clean_meds, row["diagnosis_names"])

    ddi_rows = [{"drug": h.get("drug",""), "with": ", ".join(h.get("with", [])), "type": ", ".join(h.get("type", []))}
                for h in ddi_hits]
    dci_rows = [{"drug": h.get("drug",""), "dx": ", ".join(h.get("dx", [])), "type": ", ".join(h.get("type", []))}
                for h in dci_hits]

    ddi_table = pd.DataFrame(ddi_rows) if ddi_rows else empty_ddi
    dci_table = pd.DataFrame(dci_rows) if dci_rows else empty_dci

    summary = (
        f"✅ KB loaded from: {KB_PATH}\n"
        f"KB indexed drugs: {len(KB_TEXT)}\n\n"
        f"Patient subject_id: {sid}\n"
        f"Clean meds count: {len(clean_meds)}\n"
        f"DX count: {len(to_list(row['diagnosis_names']))}\n"
        f"DDI hits: {len(ddi_hits)}\n"
        f"DCI hits: {len(dci_hits)}\n"
    )

    meds_text = "\n".join(clean_meds) if clean_meds else "(no meds after cleaning)"
    dx_text   = "\n".join(dxs_normed) if dxs_normed else "(no dx)"

    # ===== MedGemma report =====
    report_text = ""
    if use_medgemma:
        ok, msg = load_medgemma_once()
        if not ok:
            report_text = msg
        else:
            try:
                prompt = patient_prompt_ar(sid, clean_meds, dxs_normed, ddi_hits, dci_hits, max_ddi=3, max_dci=3)
                report_text = medgemma_generate(prompt)
            except Exception as e:
                report_text = f"❌ MedGemma generation error: {e}"

    return summary, meds_text, dx_text, ddi_table, dci_table, report_text

# =========================================================
# 8) UI
# =========================================================
with gr.Blocks(title="MedGuard - Conflicts + MedGemma Report") as demo:
    gr.Markdown("## MedGuard – رفع ملف الصيدلي → تعارضات → تقرير MedGemma")
    gr.Markdown(
        f"**KB_PATH:** `{KB_PATH}`  \n"
        f"**Indexed drugs:** `{len(KB_TEXT)}`  \n"
        "الملف المطلوب لازم يحتوي الأعمدة: `subject_id`, `processed_generic_meds`, `diagnosis_names`"
    )

    with gr.Row():
        file_in = gr.File(label="ارفع ملف الصيدلي (CSV/Parquet)")
        sid_in  = gr.Textbox(label="subject_id", placeholder="مثال: 10000032")
        use_mg  = gr.Checkbox(label="Generate MedGemma report", value=True)

    run_btn = gr.Button("تحليل + تقرير")

    summary_out = gr.Textbox(label="Summary", lines=10)
    with gr.Row():
        meds_out = gr.Textbox(label="Clean meds (after regex)", lines=12)
        dx_out   = gr.Textbox(label="Diagnoses (normalized)", lines=12)

    gr.Markdown("### DDI (Drug–Drug)")
    ddi_out = gr.Dataframe(interactive=False)

    gr.Markdown("### DCI (Drug–Condition)")
    dci_out = gr.Dataframe(interactive=False)

    gr.Markdown("### تقرير MedGemma (عربي)")
    report_out = gr.Textbox(label="Clinical Report", lines=16)

    run_btn.click(
        fn=analyze,
        inputs=[file_in, sid_in, use_mg],
        outputs=[summary_out, meds_out, dx_out, ddi_out, dci_out, report_out]
    )

demo.launch(share=True, debug=True, show_error=True)


In [ ]:
import os
import re
import ast
import json
import numpy as np
import pandas as pd
import gradio as gr

# ===== MedGemma imports =====
import torch
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image

# =========================================================
# 0) HuggingFace auth (Kaggle Secrets preferred)
# =========================================================
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token from Kaggle Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print("⚠️ Using HF_TOKEN from env (or empty)")

if HF_TOKEN:
    login(token=HF_TOKEN)

# =========================================================
# 1) Utils: find KB file (works on Kaggle + Local)
# =========================================================
def find_kb_file(filename="kb_openfda.parquet"):
    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for ds in os.listdir(kaggle_root):
            cand = os.path.join(kaggle_root, ds, filename)
            if os.path.exists(cand):
                return cand

    # Local fallbacks
    local_cand = os.path.join(os.getcwd(), "kb", filename)
    if os.path.exists(local_cand):
        return local_cand

    cwd_cand = os.path.join(os.getcwd(), filename)
    if os.path.exists(cwd_cand):
        return cwd_cand

    return None

KB_PATH = find_kb_file()
if KB_PATH is None:
    raise FileNotFoundError(
        "❌ Cannot find kb_openfda.parquet.\n"
        "- On Kaggle: add your KB dataset and ensure it contains kb_openfda.parquet\n"
        "- Locally: place it in ./kb/kb_openfda.parquet\n"
    )

# =========================================================
# 2) Cleaning logic (regex + stopwords)
# =========================================================
STOPWORDS = {
    "bid","tid","qid","qhs","prn","stat","daily","day","ih","qh","qpm","qam",
    "oral","po","iv","im","sc","inhalation","inh","nas","nasal",
    "tab","tabs","tablet","cap","caps","capsule","puff","spray","patch",
    "bag","unit","units","actuation",
    "mg","mcg","g","ml","l",
    "preadmission","medication","list","complete","accurate",
    "wheezing","pain","fever","nausea","sob",
    "dose"
}

def to_list(x):
    """Convert values to python list safely (supports stringified lists)."""
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple, np.ndarray)):
                    return list(v)
                return [v]
            except Exception:
                return [x]
        return [x]

    try:
        return list(x)
    except Exception:
        return [x]

def clean_processed_generics(meds):
    meds = to_list(meds)
    out = []
    for m in meds:
        s = str(m).strip().lower()
        if not s or s == "nan":
            continue
        s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
        tokens = [t for t in s.split() if t and t not in STOPWORDS]
        if tokens:
            drug = " ".join(tokens).strip()
            if len(drug) >= 4:
                out.append(drug)

    # unique while keeping order
    seen = set()
    cleaned = []
    for d in out:
        if d not in seen:
            seen.add(d)
            cleaned.append(d)
    return cleaned

def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def match_any_phrases_fast(text, phrases, max_hits=8):
    txt = norm(text)
    hits = []
    for p in phrases:
        p2 = norm(p)
        if not p2:
            continue
        if p2 in txt:
            hits.append(p2)
            if len(hits) >= max_hits:
                break
    return hits

# =========================================================
# 3) Load KB + build index (once)
# =========================================================
kb_df = pd.read_parquet(KB_PATH)

def build_kb_index(kb_df):
    kb = kb_df[["primary_generic_name","conflict_type","text"]].copy()
    kb = kb.rename(columns={
        "primary_generic_name": "drug",
        "conflict_type": "type",
        "text": "evidence"
    })

    kb["drug"] = kb["drug"].astype(str).str.lower().str.strip()
    kb["evidence"] = kb["evidence"].astype(str)
    kb = kb.dropna(subset=["drug","evidence"])

    kb_text = kb.groupby("drug")["evidence"].apply(lambda s: "\n".join(s.tolist())).to_dict()
    kb_type = kb.groupby("drug")["type"].agg(lambda s: list(pd.unique(s))).to_dict()
    return kb_text, kb_type

KB_TEXT, KB_TYPE = build_kb_index(kb_df)
print("✅ KB indexed drugs:", len(KB_TEXT))

# =========================================================
# 4) Conflict logic for ONE patient
# =========================================================
def conflicts_for_patient(clean_meds, dx_list):
    meds = [norm(m) for m in to_list(clean_meds) if norm(m)]
    dxs  = [norm(d) for d in to_list(dx_list) if norm(d)]
    dxs  = [d for d in dxs if len(d) >= 4]

    ddi_hits, dci_hits = [], []

    for drug in meds:
        txt = KB_TEXT.get(drug, "")
        if not txt:
            continue

        other_meds = [m for m in meds if m != drug]

        # DDI: KB text mentions another med from patient list
        mentioned_meds = match_any_phrases_fast(txt, other_meds, max_hits=8)
        if mentioned_meds:
            ddi_hits.append({
                "drug": drug,
                "with": mentioned_meds,
                "type": KB_TYPE.get(drug, [])
            })

        # DCI: KB text mentions a diagnosis from patient list
        mentioned_dx = match_any_phrases_fast(txt, dxs, max_hits=8)
        if mentioned_dx:
            dci_hits.append({
                "drug": drug,
                "dx": mentioned_dx,
                "type": KB_TYPE.get(drug, [])
            })

    return ddi_hits, dci_hits, meds, dxs

# =========================================================
# 5) Read pharmacist file
# =========================================================
def read_patient_file(path):
    ext = os.path.splitext(path.lower())[1]
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError("Only CSV or Parquet allowed")

REQUIRED_COLS = {"subject_id", "processed_generic_meds", "diagnosis_names"}

# =========================================================
# 6) MedGemma load + generator (lazy load)
# =========================================================
MODEL_ID = "google/medgemma-1.5-4b-it"
medgemma_model = None
medgemma_processor = None
DUMMY_IMAGE = None

def load_medgemma_once():
    global medgemma_model, medgemma_processor, DUMMY_IMAGE

    if medgemma_model is not None:
        return True, "✅ MedGemma already loaded"

    if not torch.cuda.is_available():
        return False, "❌ GPU required for MedGemma on Kaggle"

    if not HF_TOKEN:
        return False, "❌ HF_TOKEN missing (add it in Kaggle Secrets as HF_TOKEN)"

    print("⏳ Loading MedGemma...")
    medgemma_processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
    medgemma_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN,
        low_cpu_mem_usage=True
    )
    medgemma_model.eval()

    # Dummy image for "text-only" usage
    DUMMY_IMAGE = Image.fromarray(np.ones((896, 896, 3), dtype=np.uint8) * 255)
    return True, "✅ MedGemma loaded"

# STRICT prompt in English: only use provided conflicts; if none => one sentence only.
def patient_prompt_en(sid, clean_meds, dxs_normed, ddi_hits, dci_hits, max_ddi=3, max_dci=3):
    ddi = (ddi_hits or [])[:max_ddi]
    dci = (dci_hits or [])[:max_dci]

    prompt = f"""


You are a licensed clinical pharmacist.

Analyze the following patient information to identify potential:

Drug–Drug Interactions (DDIs)

Drug–Condition Interactions (DCIs)

STRICT RULES (MANDATORY):

Do NOT reveal chain-of-thought, reasoning steps, or internal analysis.

Do NOT explain how conclusions were reached.

Do NOT include intermediate steps or hidden thoughts.

Output ONLY final clinical conclusions.

REQUIRED OUTPUT FORMAT:
For each interaction, provide:

Interaction type (DDI / DCI)

Drugs / Condition involved

Severity (Minor / Moderate / Major)

Clinical risk (1–2 lines, professional)

Pharmacist recommendation (Avoid / Adjust dose / Monitor / Safe)

Use clear, concise, and professional medical language.
No commentary, no speculation, no internal reasoning.

PATIENT_ID: {sid}

CURRENT MEDS (cleaned generics, for context only):
{json.dumps(clean_meds, ensure_ascii=False)}

DIAGNOSES/CONDITIONS (for context only):
{json.dumps(dxs_normed, ensure_ascii=False)}

INTERACTIONS (the ONLY source of truth):
DDI:
{json.dumps(ddi, ensure_ascii=False)}

DCI:
{json.dumps(dci, ensure_ascii=False)}

If interactions exist, output STRICTLY in this format (no extra text):
1) SUMMARY: one short paragraph summarizing key risks based on the lists.
2) BULLETS: bullet points, one per important interaction: (interaction + brief mechanism + recommended action).
3) OVERALL RISK LEVEL: High / Medium / Low
No disclaimers.
"""
    return prompt.strip()

def medgemma_generate(prompt, max_new_tokens=260):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": DUMMY_IMAGE},
            {"type": "text", "text": prompt}
        ]
    }]

    inputs = medgemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(medgemma_model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = medgemma_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=1
        )

    text = medgemma_processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    del inputs, outputs
    torch.cuda.empty_cache()
    return text

# =========================================================
# 7) Main handler (conflicts + optional report)
#    IMPORTANT: MedGemma runs ONLY if there ARE conflicts.
# =========================================================
def analyze(file_obj, subject_id, use_medgemma):
    empty_ddi = pd.DataFrame(columns=["drug","with","type"])
    empty_dci = pd.DataFrame(columns=["drug","dx","type"])

    if file_obj is None:
        return "❌ Upload the pharmacist file first.", "", "", empty_ddi, empty_dci, ""

    path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)

    try:
        df = read_patient_file(path)
    except Exception as e:
        return f"❌ Failed to read file: {e}", "", "", empty_ddi, empty_dci, ""

    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        return f"❌ Missing columns: {sorted(missing)}", "", "", empty_ddi, empty_dci, ""

    try:
        sid = int(str(subject_id).strip())
    except Exception:
        return "❌ subject_id must be an integer.", "", "", empty_ddi, empty_dci, ""

    sub = df.loc[df["subject_id"] == sid]
    if sub.empty:
        return f"❌ subject_id={sid} not found in file.", "", "", empty_ddi, empty_dci, ""

    row = sub.iloc[0]
    clean_meds = clean_processed_generics(row["processed_generic_meds"])
    ddi_hits, dci_hits, meds_normed, dxs_normed = conflicts_for_patient(clean_meds, row["diagnosis_names"])

    ddi_rows = [{"drug": h.get("drug",""), "with": ", ".join(h.get("with", [])), "type": ", ".join(h.get("type", []))}
                for h in ddi_hits]
    dci_rows = [{"drug": h.get("drug",""), "dx": ", ".join(h.get("dx", [])), "type": ", ".join(h.get("type", []))}
                for h in dci_hits]

    ddi_table = pd.DataFrame(ddi_rows) if ddi_rows else empty_ddi
    dci_table = pd.DataFrame(dci_rows) if dci_rows else empty_dci

    has_conflicts = (len(ddi_hits) + len(dci_hits)) > 0

    summary = (
        f"✅ KB loaded from: {KB_PATH}\n"
        f"KB indexed drugs: {len(KB_TEXT)}\n\n"
        f"Patient subject_id: {sid}\n"
        f"Clean meds count: {len(clean_meds)}\n"
        f"DX count: {len(to_list(row['diagnosis_names']))}\n"
        f"DDI hits: {len(ddi_hits)}\n"
        f"DCI hits: {len(dci_hits)}\n"
    )

    meds_text = "\n".join(clean_meds) if clean_meds else "(no meds after cleaning)"
    dx_text   = "\n".join(dxs_normed) if dxs_normed else "(no dx)"

    # ===== MedGemma report =====
    report_text = ""
    if use_medgemma:
        if not has_conflicts:
            report_text = "✅ No interactions were detected based on the Knowledge Base."
        else:
            ok, msg = load_medgemma_once()
            if not ok:
                report_text = msg
            else:
                prompt = patient_prompt_en(sid, clean_meds, dxs_normed, ddi_hits, dci_hits, max_ddi=3, max_dci=3)
                report_text = medgemma_generate(prompt)

    return summary, meds_text, dx_text, ddi_table, dci_table, report_text

# =========================================================
# 8) UI
# =========================================================
with gr.Blocks(title="MedGuard - Conflicts + MedGemma Report") as demo:
    gr.Markdown("## MedGuard – Upload Pharmacist File → Conflicts → MedGemma Report")
    gr.Markdown(
        f"**KB_PATH:** `{KB_PATH}`  \n"
        f"**Indexed drugs:** `{len(KB_TEXT)}`  \n"
        "Required columns: `subject_id`, `processed_generic_meds`, `diagnosis_names`"
    )

    with gr.Row():
        file_in = gr.File(label="Upload Patient file (CSV/Parquet)")
        sid_in  = gr.Textbox(label="subject_id", placeholder="Enter Patient ID")
        use_mg  = gr.Checkbox(label="Generate MedGemma report (only if conflicts exist)", value=True)

    run_btn = gr.Button("Analyze")

    summary_out = gr.Textbox(label="Summary", lines=10)
    with gr.Row():
        meds_out = gr.Textbox(label="Clean meds (after regex)", lines=12)
        dx_out   = gr.Textbox(label="Diagnoses (normalized)", lines=12)

    gr.Markdown(" DDI (Drug–Drug)")
    ddi_out = gr.Dataframe(interactive=False)

    gr.Markdown(" DCI (Drug–Condition)")
    dci_out = gr.Dataframe(interactive=False)

    gr.Markdown(" MedGemma Report ")
    report_out = gr.Textbox(label="Clinical Report", lines=16)

    run_btn.click(
        fn=analyze,
        inputs=[file_in, sid_in, use_mg],
        outputs=[summary_out, meds_out, dx_out, ddi_out, dci_out, report_out]
    )

# Kaggle-friendly launch
demo.launch(share=True, debug=True, show_error=True)


2026-02-04 15:18:38.708697: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770218318.898797      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770218318.953361      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770218319.431521      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770218319.431563      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770218319.431565      55 computation_placer.cc:177] computation placer alr

✅ HF token from Kaggle Secrets
✅ KB indexed drugs: 3659
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a07a44d22b40468378.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
